In [1]:
%load_ext autoreload
%autoreload 2

import itertools
from math import prod

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


from assignment_1.modules.models import fit_bpr, bpr_scorer
from assignment_1.modules.util import (
    calc_pair_accuracy,
    load_csv,
    random_split,
    make_pair_validation,
)

DATA_DIR = "../data"  # notebook runs from notebooks/

In [2]:
train = load_csv("tasks_1_2_train.csv", DATA_DIR)
test = load_csv("task_2_test.csv", DATA_DIR)

In [3]:
train.head(5)

,user,item,timestamp,rating
0,u6615992338,i4026940741,1387439652,4
1,u1474879069,i5361500840,1358838765,5
2,u1060791114,i1810200320,1429854010,5
3,u1309139104,i4536388280,1409468967,4
4,u3188006761,i2217243877,1439792233,3


In [4]:
test.head(5)

,user,item1,item2,which
0,u6589235933,i6679693485,i3513531206,NaN
1,u5036346542,i9742011999,i4000434505,NaN
2,u2543223670,i8553129811,i8159127157,NaN
3,u6680502243,i5391486098,i3623313194,NaN
4,u2041216719,i5735921315,i5729684422,NaN


In [5]:
# interactions per item in training
pop = train["item"].value_counts()           
p1 = test["item1"].map(pop).fillna(0)
p2 = test["item2"].map(pop).fillna(0)

print(f"unseen users: {(~test.user.isin(train.user)).mean():.1%}")
print(f"unseen item1: {(p1 == 0).mean():.1%} | unseen item2: {(p2 == 0).mean():.1%}")
print(f"median popularity: item1 = {p1.median():.0f}, item2 = {p2.median():.0f}")
print(f"P(item1 more popular) = {(p1 > p2).mean():.3f} | ties = {(p1 == p2).mean():.3f}")

print("\nreference medians:")
print(f"  item drawn uniformly from catalog: {pop.median():.0f}")
print(f"  item drawn ∝ interactions:         {train['item'].map(pop).median():.0f}")

unseen users: 0.0%
unseen item1: 0.0% | unseen item2: 0.0%
median popularity: item1 = 58, item2 = 57
P(item1 more popular) = 0.497 | ties = 0.007

reference medians:
  item drawn uniformly from catalog: 18
  item drawn ∝ interactions:         59


In [6]:
tr, val = random_split(train, val_frac=0.10, seed=0)
seen = val.user.isin(tr.user) & val.item.isin(tr.item)
val = val[seen].reset_index(drop=True)

print(f"kept {seen.mean():.1%} of validation rows")

kept 100.0% of validation rows


In [7]:
tr, pairs = make_pair_validation(train, seed=0)
calc_pair_accuracy(pairs, bpr_scorer(fit_bpr(tr, factors=5, regularization=0.01, learning_rate=0.01, iterations=100)))

pairs.head()

,user,item1,item2,which
0,u1603088681,i9960281822,i3791430712,i9960281822
1,u3898574929,i9464761793,i0695900379,i9464761793
2,u7753644730,i8642457945,i5114999221,i5114999221
3,u2943949681,i4451239709,i2201875856,i4451239709
4,u8122465982,i1240061652,i9674039066,i1240061652


In [8]:
rng = np.random.default_rng(0)
pop = train.item.value_counts()

calc_pair_accuracy(pairs, lambda u, i: rng.random(len(i)))                       # random     ≈ 0.50

0.498721557336542

In [9]:
calc_pair_accuracy(pairs, lambda u, i: np.zeros(len(i)))                         # constant   ≈ 0.50 (all ties -> item1)

0.5018478330934015

In [10]:
calc_pair_accuracy(pairs, lambda u, i: pd.Series(i).map(pop).fillna(0).to_numpy()) 

0.5056401882211383

In [11]:
fit = fit_bpr(tr, factors=5, regularization=0.01, learning_rate=0.01, iterations=100)
# ≈ 0.877
base = calc_pair_accuracy(pairs, bpr_scorer(fit))   
print(base)

0.8771083560731399


In [12]:
grid1 = {
    "factors":        [5, 20, 50],
    "regularization": [0.001, 0.01, 0.1],
    "learning_rate":  [0.01],
    "iterations":     [100],
}

grid2 = {
    "factors":        [50, 100, 200],
    "regularization": [0.003, 0.01, 0.03],
    "learning_rate":  [0.01],
    "iterations":     [100],
}

rows = []
for values in tqdm(itertools.product(*grid2.values()), total=prod(map(len, grid2.values()))):
    params = dict(zip(grid2, values))          # {"factors": 20, "regularization": 0.01, ...}
    fit = fit_bpr(tr, **params)
    acc = calc_pair_accuracy(pairs, bpr_scorer(fit))
    rows.append({**params, "val_acc": acc, "vs_base_%": (acc / base - 1) * 100})

results = pd.DataFrame(rows).sort_values("val_acc", ascending=False)
results

  0%|          | 0/9 [00:00<?, ?it/s]

,factors,regularization,learning_rate,iterations,val_acc,vs_base_%
3,100,0.003,0.01,100,0.914742,4.290631
6,200,0.003,0.01,100,0.914382,4.249599
0,50,0.003,0.01,100,0.912271,4.008917
1,50,0.010,0.01,100,0.911841,3.959923
4,100,0.010,0.01,100,0.911556,3.927465
7,200,0.010,0.01,100,0.910412,3.797019
2,50,0.030,0.01,100,0.899599,2.564212
5,100,0.030,0.01,100,0.898783,2.471124
8,200,0.030,0.01,100,0.897386,2.311894
